# **Laboratorio guiado: Ataque de Padding Oracle en AES-CBC**

## **Importación de Paquetes y Librerías**

In [2]:
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes
from Crypto.Util.Padding import pad , unpad
from VulnerableServer import VulnerableServer

## **Recover Block**

La función `recover_block` implementa el núcleo del ataque de Padding Oracle sobre el modo AES-CBC. Su objetivo es recuperar el bloque de texto plano $M_i$ correspondiente a un bloque cifrado $C_{curr}$, utilizando el bloque anterior $C_{prev}$ como IV en las consultas al oráculo.
El proceso consiste en modificar los bytes de $C_{prev}$ para cada posición del bloque, de derecha a izquierda, hasta encontrar un valor que produzca un padding válido según el oráculo. Una vez encontrado, se calcula el estado intermedio y el byte de texto plano correspondiente. El procedimiento se repite para cada byte del bloque, hasta reconstruir el bloque completo.
Esta técnica permite descifrar bloques cifrados sin conocer la clave, aprovechando la información proporcionada por el oráculo de padding.

In [3]:
def recover_block(oracle, C_prev, C_curr):
    """
    Recupera un bloque de texto plano M_i correspondiente a C_curr,
    usando C_prev como bloque anterior.
    
    Args:
        oracle: función que devuelve True/False según validez del padding
        C_prev: bloque anterior (16 bytes)
        C_curr: bloque actual a descifrar (16 bytes)
    
    Returns:
        bytes: bloque de texto plano M_i (16 bytes)
    """
    BLOCK = 16
    
    I = [0] * BLOCK  # Estado intermedio D_k(C_i)
    M = [0] * BLOCK  # Texto plano resultante
    
    consultas = 0  # Contador de consultas para métricas
    
    # Para cada posición de padding p = 1, 2, ..., 16
    for p in range(1, BLOCK + 1):
        # Índice activo (de derecha a izquierda)
        t = BLOCK - p
        
        # Preparar sufijo con invariante de padding
        c_prev_mod = bytearray(C_prev)
        
        # Establecer bytes ya conocidos para formar padding válido
        for j in range(t + 1, BLOCK):
            c_prev_mod[j] = I[j] ^ p
        
        # Barrido para encontrar el byte correcto
        found = False
        for g in range(256):
            consultas += 1
            
            # Modificar byte activo
            c_prev_mod[t] = g
            
            # Crear payload de 2 bloques
            payload = bytes(c_prev_mod) + C_curr
            
            # Consultar oracle
            if oracle(payload):
                # Encontrado! Calcular estado intermedio y texto plano
                I[t] = g ^ p
                M[t] = I[t] ^ C_prev[t]
                found = True
                break
        
        if not found:
            raise Exception(f"No se pudo encontrar valor válido para posición {t}")
    
    print(f"  Bloque recuperado con {consultas} consultas")
    return bytes(M)


## **Función Recover Message**
La función `recover_message` permite recuperar el mensaje original a partir de un ciphertext cifrado en modo AES-CBC, utilizando un oráculo de padding. El procedimiento consiste en dividir el ciphertext en bloques, y aplicar el ataque de Padding Oracle sobre cada bloque cifrado, empleando el bloque anterior como IV. El resultado es la reconstrucción completa del texto plano, eliminando el padding PKCS#7 al final. Esta técnica demuestra cómo una vulnerabilidad en el manejo del padding puede comprometer la confidencialidad de los datos cifrados.

In [4]:
def recover_message(oracle, ct):
    """
    Recupera el mensaje completo dado un ciphertext.
    
    Args:
        oracle: función que devuelve True/False según validez del padding
        ct: ciphertext completo (IV || C1 || C2 || ... || Cl)
    
    Returns:
        bytes: mensaje original sin padding
    """
    BLOCK = 16
    if len(ct) < 2 * BLOCK or len(ct) % BLOCK != 0:
        raise ValueError("Ciphertext debe tener al menos 2 bloques y longitud múltiplo de 16")
    
    # Separar en bloques
    blocks = [ct[i:i+BLOCK] for i in range(0, len(ct), BLOCK)]
    iv = blocks[0]  # C_0 = IV
    cblocks = blocks[1:]  # C_1, C_2, ..., C_l
    
    print(f"Recuperando mensaje de {len(cblocks)} bloques...")
    
    # Recuperar cada bloque
    recovered = b""
    for i, curr_block in enumerate(cblocks):
        print(f"Recuperando bloque {i+1}/{len(cblocks)}...")
        
        # El bloque anterior es IV (para el primer bloque) o el bloque cifrado anterior
        prev_block = iv if i == 0 else blocks[i]
        
        # Recuperar bloque actual
        block_plaintext = recover_block(oracle, prev_block, curr_block)
        recovered += block_plaintext
    
    # Remover padding PKCS#7
    try:
        unpadded = unpad(recovered, BLOCK)
        return unpadded
    except ValueError as e:
        print(f"Error al remover padding: {e}")
        print(f"Datos recuperados (hex): {recovered.hex()}")
        return recovered


## **Demostración del ataque Padding Oracle**

In [ ]:
"""
TODO: Revisar por qué el ataque no funciona cuando el mensaje contiene dos puntos (:)
"""
server = VulnerableServer()

test_messages = [
    b"Attack at dawn!!",
    b"Meet me at the park at 9pm.",
    b"Top secret message The eagle has landed.",
    b"Padding oracle attacks are fun!",
    b"Short",
    b"A" * 16,
    b"B" * 31,
    b"C" * 32,
]

for i, msg in enumerate(test_messages):
    print(f"\n--- Test {i+1}: Mensaje original: {msg} ---")
    
    # Cifrar mensaje
    ct = server.encrypt(msg)
    print(f"Ciphertext (hex): {ct.hex()}")
    print(f"Longitud del ciphertext: {len(ct)} bytes")
    
    # Recuperar mensaje usando el ataque
    try:
        recovered = recover_message(server.decrypt, ct)
    except Exception as e:
        print(f"Error durante la recuperación: {e}")
        continue

    print(f"Mensaje recuperado: {recovered}")
    
    # Verificar éxito
    if recovered == msg:
        print("El mensaje fue correctamente recuperado.")
    else:
        print("Fallo. El mensaje recuperado no coincide con el original.")


--- Test 1: Mensaje original: b'Attack at dawn!!' ---
Ciphertext (hex): 328a809bff0b78c5b5089dabe3c8388108fddfc669d4e791162b2325683d5f165978200ca13b6ab521fc9f736b3080ba
Longitud del ciphertext: 48 bytes
Recuperando mensaje de 2 bloques...
Recuperando bloque 1/2...
  Bloque recuperado con 2566 consultas
Recuperando bloque 2/2...
  Bloque recuperado con 1804 consultas
Mensaje recuperado: b'Attack at dawn!!'
El mensaje fue correctamente recuperado.

--- Test 2: Mensaje original: b'Meet me at the park at 9pm.' ---
Ciphertext (hex): ae26770cbff5845df365fae9474c2dd26a24f23ad645975cf6ef3fde74050217f6ae20e2aae6c14b68d68733f652d32f
Longitud del ciphertext: 48 bytes
Recuperando mensaje de 2 bloques...
Recuperando bloque 1/2...
  Bloque recuperado con 1936 consultas
Recuperando bloque 2/2...
  Bloque recuperado con 1512 consultas
Mensaje recuperado: b'Meet me at the park at 9pm.'
El mensaje fue correctamente recuperado.

--- Test 3: Mensaje original: b'Top secret message The eagle has landed.' -